In [0]:
%pip install langchain-text-splitters==1.0 -q

In [0]:
import pandas as pd
import pyspark.sql.functions as sf
import pyspark.sql.types as st
import json
from typing import Generator, Iterator

from langchain_text_splitters import RecursiveCharacterTextSplitter

table_parsed = "workspace.sephora_products_and_skincare_reviews.bronze_pdf_parsed"
table_chunked = "workspace.sephora_products_and_skincare_reviews.bronze_pdf_chunked"

ENDPOINT = "databricks-gpt-oss-20b"

prompt_prefix = """
You are a helpful assistant. Given JSON object representing a parsed document
(with pages, elements, and metadata), convert the content into clean, readable markdown.
Use "== page ==" to separate each page. Preserve important structure such as headers,
tables, and captions. Do not include any JSON or code blocks in the output-just
the clean markdown text.

JSON:

"""

In [0]:
df_parsed = spark.read.table(table_parsed)

df_transformed = (
    df_parsed.withColumn(
        "clean_markdown_text",
        sf.expr(f"""
            ai_query(
                '{ENDPOINT}',
                CONCAT('{prompt_prefix}', CAST(parsed_content AS STRING)),
                responseFormat => '{{"type": "text"}}'
            )
        """)
    )
)

In [0]:
display(df_transformed)

In [0]:
def _page_id_from_bbox(bbox: list[dict] | dict | None) -> int | None:
    """Return bbox.page_id if present."""
    if not bbox or not (isinstance(bbox, list) or isinstance(bbox, dict)):
        return None
    elif isinstance(bbox, list):
        bbox = bbox[0]
    return bbox.get("page_id")

def extract_contents_from_json(json_str: str) -> str:
    try: 
        content = json.loads(json_str)

        document = content.get("document", content)
        elements = document.get("elements", []) if isinstance(document, dict) else []
        if not isinstance(elements, list):
            return ""
        
        out_lines = []
        current_page = None

        for el in elements:
            if not isinstance(el, dict):
                continue

            pid = _page_id_from_bbox(el.get("bbox"))
            if pid and current_page and pid != current_page:
                out_lines.append(f"")
                out_lines.append(f"== page ==")
                out_lines.append(f"")
            if pid:
                current_page = pid
            
            c = el.get("content")
            if not(isinstance(c, str) and c.strip()):
                c = el.get("description")
            if isinstance(c, str):
                out_lines.append(c)

                t = (el.get("type") or "").lower()
                if t != "text":
                    out_lines.append("")
        
        return "\n".join(out_lines)
    except Exception:
        return ""


def extract_contents_udf():
    """A small factory to create a PySpark UDF that uses the function above."""
    @sf.udf(st.StringType())
    def _udf(json_str):
        try:
            return extract_contents_from_json(json_str)
        except Exception as e:
            return f"Error: {e}"
    return _udf


In [0]:
safe_json_col = sf.coalesce(
    sf.to_json(
        sf.col("parsed_content")
    ),
    sf.col("parsed_content").cast("string")
)

df_plain_text = df_parsed.withColumn(
    "plain_text",
    extract_contents_udf()(safe_json_col)
)

display(df_plain_text.select("path", "plain_text"))

In [0]:
CHUNK_SIZE = 2000
CHUNK_OVERLAP = 200

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n== page ==\n", "== page ==", "\n\n", "\n", " ", ""]
)

schema = st.StructType([
    st.StructField("path", st.StringType(), True),
    st.StructField("chunk", st.StringType(), True),
])

def split_rows(iterator: Iterator) -> Generator[pd.DataFrame, None, None]:
    for pdf in iterator:
        out = []
        for _, row in pdf.iterrows():
            path = row["path"]
            text = row["plain_text"]
            if isinstance(text, str) and text.strip():
                for c in splitter.split_text(text):
                    out.append((path, c))
        yield pd.DataFrame(out, columns=["path", "chunk"])


df_chunks = (
    df_plain_text
    .select(
        "path",
        "plain_text"
    )
    .mapInPandas(split_rows, schema=schema)
)

display(df_chunks)

In [0]:
df_chunks = df_chunks.withColumn(
    "id",
    sf.monotonically_increasing_id()
)

In [0]:
(
    df_chunks
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(table_chunked)
)